In [1]:
import os
import requests
import re
from dotenv import load_dotenv
from hsclient import HydroShare
import rdflib
import json
import ast
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from typing import Dict, Any, List, Optional, Tuple
load_dotenv()

True

In [4]:
def authenticate_hydroshare():
    hs_user = os.getenv('HYDROSHARE_USERNAME')
    hs_pass = os.getenv('HYDROSHARE_PASSWORD')

    if not hs_user or not hs_pass:
        raise EnvironmentError("System Error: HydroShare credentials not found in environment variables.")

    # Bypasses the interactive prompt by passing credentials directly
    hs = HydroShare(username=hs_user, password=hs_pass)
    
    return hs

In [5]:
hs_session = authenticate_hydroshare()
print("Authentication layer verified.")

Authentication layer verified.


In [6]:
res_test = "27045581bdea4808a393330f2417379c"
res_test_obj = hs_session.resource(res_test)
print(res_test_obj)

/resource/27045581bdea4808a393330f2417379c/data/resourcemap.xml


In [8]:
def download_raw_corpus(hs_client, resource_id: str, base_path: str = "./ciroh_corpus"):
    """
    Downloads the 'Level Zero' artifacts for a HydroShare resource to local disk.
    Includes explicit exception handling for hsclient's empty file index bug.
    """
    res_dir = os.path.join(base_path, resource_id)
    os.makedirs(res_dir, exist_ok=True)
    
    try:
        res = hs_client.resource(resource_id)
        
        # 1. Download Native Pydantic Dump (meta)
        try:
            meta_dict = res.metadata.model_dump() if hasattr(res.metadata, 'model_dump') else res.metadata.dict()
            with open(os.path.join(res_dir, "meta_dump.json"), 'w', encoding='utf-8') as f:
                json.dump(meta_dict, f, default=str, indent=4)
        except Exception as e:
            print(f"  -> Warning: Failed to dump meta for {resource_id}: {e}")

        # 2. Download Native System Metadata (sys_meta)
        try:
            with open(os.path.join(res_dir, "sys_meta_dump.json"), 'w', encoding='utf-8') as f:
                json.dump(res.system_metadata(), f, indent=4)
        except Exception as e:
            print(f"  -> Warning: Failed to dump sys_meta for {resource_id}: {e}")

        # 3. Download Raw XML/RDF 
        try:
            raw_xml = hs_client._hs_session.retrieve_string(res.metadata_path)
            with open(os.path.join(res_dir, "raw_metadata.xml"), 'w', encoding='utf-8') as f:
                f.write(raw_xml)
        except Exception as e:
            print(f"  -> Warning: Failed to download raw XML for {resource_id}: {e}")

        # 4. Download Physical README and capture file inventory
        file_inventory = []
        try:
            for f in res.files():
                file_inventory.append({
                    "file_name": str(f.name),
                    "file_path": str(f.path),
                    "extension": str(f.extension),
                    "checksum": str(f.checksum)
                })
                
                if str(f.name).lower().startswith("readme") and str(f.extension).lower() in ['.md', '.txt']:
                    try:
                        content = hs_client._hs_session.retrieve_string(f.url)
                        with open(os.path.join(res_dir, f.name), 'w', encoding='utf-8') as readme_f:
                            readme_f.write(content)
                    except Exception as e:
                        print(f"  -> Warning: Failed to download {f.name}: {e}")
                        
        except IndexError:
            # Crucial control structure for CollectionResources and ToolResources
            print(f"  -> Warning: No physical files found (IndexError mitigated).")
        except Exception as file_e:
             print(f"  -> Warning: File inventory failed: {file_e}")

        with open(os.path.join(res_dir, "file_inventory.json"), 'w', encoding='utf-8') as f:
             json.dump(file_inventory, f, indent=4)

        return True

    except Exception as e:
        print(f"Critical Failure downloading {resource_id}: {str(e)}")
        return False

In [9]:
# Define file paths
csv_file_path = 'HydroShare_IDs_03272026.csv'
#csv_file_path = 'HydroShare_Failed_IDs_03272026.csv'
base_corpus_path = './ciroh_corpus'

# 1. Read and validate IDs from the CSV file
ciroh_corpus_ids = []

try:
    with open(csv_file_path, 'r', encoding='utf-8') as file:
        for line in file:
            # Strip whitespace and newline characters
            clean_id = line.strip()
            
            # Validate standard HydroShare ID length (32 characters)
            if len(clean_id) == 32:
                ciroh_corpus_ids.append(clean_id)
            elif clean_id:
                print(f"Warning: Anomalous ID format skipped -> {clean_id}")
                
except FileNotFoundError:
    print(f"Error: File '{csv_file_path}' not found. Please check the directory path.")

print(f"Total valid IDs to process: {len(ciroh_corpus_ids)}\n")
print("-" * 50)

# 2. Orchestrate the Level Zero baseline download
success_count = 0
failed_ids = []

# Note: hs_session must be an authenticated HydroShare instance in your environment
for index, res_id in enumerate(ciroh_corpus_ids, start=1):
    print(f"[{index}/{len(ciroh_corpus_ids)}] Processing resource: {res_id}")
    
    # Execute the extraction function defined previously
    success = download_raw_corpus(hs_session, res_id, base_path=base_corpus_path)
    
    if success:
        success_count += 1
    else:
        failed_ids.append(res_id)

# 3. Execution reporting
print("\n" + "=" * 50)
print("CORPUS ACQUISITION REPORT")
print("=" * 50)
print(f"Total processed: {len(ciroh_corpus_ids)}")
print(f"Successful downloads: {success_count}")
print(f"Failed downloads: {len(failed_ids)}")

if failed_ids:
    print("\nList of failed IDs (check network connection or access permissions):")
    for fid in failed_ids:
        print(f" - {fid}")

Total valid IDs to process: 42

--------------------------------------------------
[1/42] Processing resource: 7d960b7fdfee480895fd845bade1b75a
[2/42] Processing resource: 0ef4366e7711478fa2637f5049b4881a
[3/42] Processing resource: 3200bab682ec4c3287147cbe40e768ee
[4/42] Processing resource: a8b243957f7946e388d10ab206990675
[5/42] Processing resource: 88e0ebf2719c492381efcb27fba71032
[6/42] Processing resource: 18134d8103d84deeb2a9a5008068158d
[7/42] Processing resource: 1a2e115c212f4f4a80660f94339205e6
[8/42] Processing resource: 7477452a93e04b1099a3879353c2253a
[9/42] Processing resource: 0d0a8d67a7fe4676bc84f6973dde0fc8
[10/42] Processing resource: fc8539358fe64ca6a47468728a0687a1
[11/42] Processing resource: 48b3355015f84e98a2cc6b6d134e77f2
[12/42] Processing resource: 023bfa101586432ba0f6ed9ddfea60a9
[13/42] Processing resource: 23b51cb99b5445a1af740a21c5acaecb
[14/42] Processing resource: 2dd1ac86e8854d4fb9fe5fbafaec2b98


https://ciroh.awi.2i2c.cloud/hub/spawn?next=/user-redirect/nbfetch/hs-pull?id=${HS_RES_ID}%26app%3Dlab%0A does not look like a valid URI, trying to serialize this will break.
https://ciroh.awi.2i2c.cloud/hub/spawn?next=/user-redirect/nbfetch/hs-pull?id=${HS_RES_ID}%26start%3D${HS_FILE_PATH}%26app%3Dlab does not look like a valid URI, trying to serialize this will break.


[15/42] Processing resource: 5caaa9919c2e490eb735aebc5e741c35
[16/42] Processing resource: 0b78e7735f2c4e858f3247f740dcd30f
  -> Warning: No physical files found (IndexError mitigated).
[17/42] Processing resource: 4b5d7c4a47da4e05bc6507b3071d1c82
[18/42] Processing resource: 72ea9726187e43d7b50a624f2acf591f
[19/42] Processing resource: 6ca065138d764339baf3514ba2f2d72f
[20/42] Processing resource: 631be88704ef4167b12e9ad9d2529ba9
[21/42] Processing resource: 56e0ac25249d4451845a843eb3154954
[22/42] Processing resource: c738c05278a34bc9848dd14d61cffab9
[23/42] Processing resource: 5d88721054a5458d93ad837192f38abb
[24/42] Processing resource: 302dcbef13614ac486fb260eaa1ca87c
  -> Warning: No physical files found (IndexError mitigated).
[25/42] Processing resource: ff4e9c4e87ef4d7d923efe77f5ed2b83
[26/42] Processing resource: ec43b15ed20744f39bc757f7b2e4d51e
[27/42] Processing resource: 75e311b73ba64f259ec52668e458ca99
[28/42] Processing resource: 4d98edaf289942e49ba1e2491d123a32
[29/42] 

In [2]:
def available_metadata(hs_client: HydroShare, resource_id: str):
    """
    List all available metadata fields for a given resource ID, including both standard and system metadata.
    This function is used for exploratory purposes to understand the metadata structure of HydroShare resources.
    """
    try:
        res = hs_client.resource(resource_id)
        meta = res.metadata
        sys_meta = res.system_metadata()

        # 0. List of all available attributes

        print("--- Introspection: res (Client Controller Object) ---")
        print("Type:", type(res))
        print("Public Methods & Attributes:")
        print([attr for attr in dir(res) if not attr.startswith('_')])

        print("\n---------------------------------------------------------------------\n")

        print("--- Introspection: meta (Pydantic Data Model) ---")
        print("Type:", type(meta))
        print("Populated Attributes (Keys):")
        try:
            print(list(meta.model_dump().keys())) 
        except AttributeError:
            print(list(meta.dict().keys()))

        # Diagnostic Block: Inspecting deep attributes
        print("--- Diagnostic: Additional Metadata ---")
        if hasattr(meta, 'additional_metadata') and meta.additional_metadata:
            print(meta.additional_metadata)
        else:
            print("No additional metadata present.")

        print("\n--- Diagnostic: Relations ---")
        if hasattr(meta, 'relations') and meta.relations:
            for rel in meta.relations:
                print(f"Type: {rel.type}, Value: {rel.value}")
        else:
            print("No relations present.")

        print("\n---------------------------------------------------------------------\n")
        
        print("--- Introspection: sys_meta (Standard Python Dictionary) ---")
        print("Type:", type(sys_meta))
        print("Populated Attributes (Keys):")
        if isinstance(sys_meta, dict):
            print(list(sys_meta.keys()))
        else:
            print("Unexpected type for sys_meta.")

        print("\n---------------------------------------------------------------------\n")        


    except Exception as e:
        print(f"Failed to extract metadata for resource {resource_id}: {str(e)}")
        return {"resource_id": resource_id, "error": str(e)}

In [ ]:
test_id = '0580878f98d548b08923b780e667c396' 

available_metadata(hs_session, test_id)

In [15]:
NS = {
    "dc": "http://purl.org/dc/elements/1.1/",
    "dcterms": "http://purl.org/dc/terms/",
    "hsterms": "https://www.hydroshare.org/terms/",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
}

GEOSPATIAL_EXTENSIONS = {
    ".tif", ".tiff", ".vrt", ".shp", ".gpkg", ".nc", ".geojson", ".json"
}

HIDDEN_CHARS_RE = re.compile(r"[\u00ad\u200b\u200c\u200d\ufeff]")
HS_RESOURCE_ID_RE = re.compile(r"/resource/([0-9a-fA-F]{32})")
URL_RE = re.compile(r"https?://[^\s<>\"]+")

README_ALLOWED_SUFFIXES = {"readme.md", "readme.txt"}
README_HEADING_RE = re.compile(r"^(#{1,6})\s+(.*\S)\s*$")
README_RULE_RE = re.compile(r"^\s*([-*_])\1{2,}\s*$")

def _clean_text(value: Optional[str]) -> Optional[str]:
    if value is None:
        return None
    if not isinstance(value, str):
        return value
    value = HIDDEN_CHARS_RE.sub("", value)
    value = value.replace("\r\n", "\n").replace("\r", "\n")
    value = re.sub(r"[ \t]+", " ", value)
    value = re.sub(r"\n{3,}", "\n\n", value)
    value = value.strip()
    return value or None


def _clean_list(values: List[str]) -> List[str]:
    seen = set()
    result = []
    for v in values:
        v = _clean_text(v)
        if v and v not in seen:
            seen.add(v)
            result.append(v)
    return result


def _extract_hydroshare_resource_id(text: Optional[str]) -> Optional[str]:
    if not text or not isinstance(text, str):
        return None
    match = HS_RESOURCE_ID_RE.search(text)
    return match.group(1).lower() if match else None


def _normalize_extracted_url(url: str) -> str:
    if not isinstance(url, str):
        return url
    # Quita puntuación de cierre que suele quedar pegada en citas o texto corrido
    return url.rstrip(".,;:)]}>\"'")


def _extract_urls(text: Optional[str]) -> List[str]:
    if not text or not isinstance(text, str):
        return []

    urls = []
    seen = set()

    for raw_url in URL_RE.findall(text):
        clean_url = _normalize_extracted_url(raw_url)
        if clean_url and clean_url not in seen:
            urls.append(clean_url)
            seen.add(clean_url)
    return urls


def _summarize_data_url(data_url: Optional[str]) -> Dict[str, Any]:
    if not data_url or not isinstance(data_url, str):
        return {
            "present": False,
            "media_type": None,
            "char_length": 0
        }

    media_type = None
    if data_url.startswith("data:"):
        try:
            media_type = data_url.split(":", 1)[1].split(";", 1)[0]
        except Exception:
            media_type = None

    return {
        "present": True,
        "media_type": media_type,
        "char_length": len(data_url)
    }


def _looks_like_url(value: Optional[str]) -> bool:
    return isinstance(value, str) and value.startswith(("http://", "https://"))


def _read_json(path: str, default: Any) -> Any:
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def _normalize_readme_text_for_dedup(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\x00", "")
    return text.strip()


def _clean_readme_section_body(section_lines: List[str]) -> str:
    cleaned = []
    for ln in section_lines:
        if README_RULE_RE.match(ln):
            continue
        cleaned.append(ln)
    return "\n".join(cleaned).strip()


def _find_readme_candidates(res_dir: str) -> List[Tuple[str, str]]:
    """
    Returns unique README files found in the resource directory.
    Deduplicates by:
      1) real path
      2) normalized content
    """
    results = []
    seen_real_paths = set()
    seen_contents = set()

    for name in sorted(os.listdir(res_dir), key=lambda x: x.lower()):
        path = os.path.join(res_dir, name)

        if not os.path.isfile(path):
            continue

        if name.lower() not in README_ALLOWED_SUFFIXES:
            continue

        real_path = os.path.realpath(path)
        if real_path in seen_real_paths:
            continue
        seen_real_paths.add(real_path)

        with open(path, "r", encoding="utf-8", errors="replace") as f:
            raw_text = f.read()

        normalized_text = _normalize_readme_text_for_dedup(raw_text)
        if not normalized_text:
            continue

        if normalized_text in seen_contents:
            continue
        seen_contents.add(normalized_text)

        results.append((name, raw_text))

    return results


def _parse_readme_sections(readme_text: str) -> List[Dict[str, Any]]:
    """
    Parse README text into hierarchical sections preserving markdown heading depth.
    Includes headings without body if they have children.
    """
    text = readme_text.replace("\r\n", "\n").replace("\r", "\n")
    lines = text.split("\n")

    sections: List[Dict[str, Any]] = []
    next_section_id = 1

    preamble_lines: List[str] = []
    current_section: Optional[Dict[str, Any]] = None
    stack: List[Tuple[int, int]] = []  # (heading_level, section_id)

    def flush_current_section():
        nonlocal current_section

        if current_section is None:
            return

        body = _clean_readme_section_body(current_section["lines"])

        if body or current_section.get("has_children", False):
            if not body:
                body = current_section["heading"]

            sections.append({
                "section_id": current_section["section_id"],
                "parent_section_id": current_section["parent_section_id"],
                "heading": current_section["heading"],
                "heading_level": current_section["heading_level"],
                "heading_marker": current_section["heading_marker"],
                "section_text": body,
                "has_children": bool(current_section.get("has_children", False))
            })

        current_section = None

    def create_intro_section(intro_lines: List[str]):
        nonlocal next_section_id

        intro_text = _clean_readme_section_body(intro_lines)
        if not intro_text:
            return

        sections.append({
            "section_id": next_section_id,
            "parent_section_id": None,
            "heading": "README Introduction",
            "heading_level": 0,
            "heading_marker": None,
            "section_text": intro_text,
            "has_children": False
        })
        next_section_id += 1

    for line in lines:
        heading_match = README_HEADING_RE.match(line)

        if heading_match:
            hashes = heading_match.group(1)
            heading = heading_match.group(2).strip()
            heading_level = len(hashes)

            if current_section is None and preamble_lines:
                create_intro_section(preamble_lines)
                preamble_lines = []

            if current_section is not None and heading_level > current_section["heading_level"]:
                current_section["has_children"] = True

            flush_current_section()

            while stack and stack[-1][0] >= heading_level:
                stack.pop()

            parent_section_id = stack[-1][1] if stack else None

            current_section = {
                "section_id": next_section_id,
                "parent_section_id": parent_section_id,
                "heading": heading,
                "heading_level": heading_level,
                "heading_marker": hashes,
                "lines": [],
                "has_children": False
            }

            stack.append((heading_level, next_section_id))
            next_section_id += 1

        else:
            if current_section is None:
                preamble_lines.append(line)
            else:
                current_section["lines"].append(line)

    if current_section is None and preamble_lines:
        create_intro_section(preamble_lines)
    else:
        flush_current_section()

    return sections

def _read_optional_readme(res_dir: str) -> Dict[str, Any]:
    candidates = _find_readme_candidates(res_dir)

    if not candidates:
        return {
            "readme_present": False,
            "readme_file_count": 0,
            "primary_readme_file": None,
            "readme_files": []
        }

    readme_files = []

    for source_file, raw_text in candidates:
        clean_text = _clean_text(raw_text)
        readme_sections = _parse_readme_sections(raw_text)

        readme_files.append({
            "source_file": source_file,
            "readme_text_raw": raw_text,
            "readme_text_clean": clean_text,
            "readme_char_count": len(raw_text),
            "readme_sections": readme_sections
        })

    # Prefer markdown as the primary README when available
    primary_readme_file = None
    for item in readme_files:
        if item["source_file"].lower().endswith(".md"):
            primary_readme_file = item["source_file"]
            break

    if primary_readme_file is None and readme_files:
        primary_readme_file = readme_files[0]["source_file"]

    return {
        "readme_present": True,
        "readme_file_count": len(readme_files),
        "primary_readme_file": primary_readme_file,
        "readme_files": readme_files
    }


def _split_tag(tag: str) -> Tuple[Optional[str], str]:
    if tag.startswith("{"):
        uri, local = tag[1:].split("}", 1)
        return uri, local
    return None, tag


def _qname(tag: str) -> str:
    uri, local = _split_tag(tag)
    if uri is None:
        return local
    for prefix, ns_uri in NS.items():
        if uri == ns_uri:
            return f"{prefix}:{local}"
    return local


def _predicate_uri(tag: str) -> Optional[str]:
    uri, local = _split_tag(tag)
    if uri is None:
        return None
    return f"{uri}{local}"


def _elem_value(elem: Optional[ET.Element]) -> Tuple[Optional[str], Optional[str]]:
    if elem is None:
        return None, None

    resource_attr = elem.attrib.get(f"{{{NS['rdf']}}}resource")
    if resource_attr:
        return _clean_text(resource_attr), "uri_ref"

    text = _clean_text("".join(elem.itertext()))
    if text:
        return text, "literal"

    return None, None


def _find_hsterms_value(resource_elem: ET.Element, tag_name: str) -> Tuple[Optional[str], Optional[str]]:
    parent = resource_elem.find(f"{{{NS['hsterms']}}}{tag_name}")
    if parent is None:
        return None, None

    value_elem = parent.find(f".//{{{NS['hsterms']}}}value")
    if value_elem is None:
        return None, None

    return _elem_value(value_elem)


def _find_hsterms_values(resource_elem: ET.Element, tag_name: str) -> List[str]:
    parent = resource_elem.find(f"{{{NS['hsterms']}}}{tag_name}")
    if parent is None:
        return []

    values = []
    for value_elem in parent.findall(f".//{{{NS['hsterms']}}}value"):
        value, _ = _elem_value(value_elem)
        if value:
            values.append(value)
    return _clean_list(values)


def _normalize_tool_url_template(value: Optional[str]) -> Optional[str]:
    value = _clean_text(value)
    if not value:
        return None
    return value.replace("%0A", "")


def _normalize_units(value: Any) -> Any:
    if value is None:
        return None

    # Si ya es lista, limpiarla
    if isinstance(value, list):
        return _clean_list([str(v) for v in value])

    # Si es string, intentar interpretar listas serializadas como "['Decimal degrees']"
    if isinstance(value, str):
        cleaned = _clean_text(value)
        if not cleaned:
            return None

        try:
            parsed = ast.literal_eval(cleaned)
            if isinstance(parsed, list):
                parsed_list = _clean_list([str(v) for v in parsed])
                if len(parsed_list) == 1:
                    return parsed_list[0]
                return parsed_list
        except Exception:
            pass

        return cleaned

    return value


def _build_people(people_raw: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], List[Dict[str, str]]]:
    people = []
    identifiers_flat = []

    for person in people_raw:
        if not isinstance(person, dict):
            continue

        identifiers = person.get("identifiers", {}) or {}
        person_identifiers = []

        for id_type, url in identifiers.items():
            url = _clean_text(url)
            if url:
                item = {
                    "id_type": _clean_text(id_type),
                    "url": url
                }
                person_identifiers.append(item)
                identifiers_flat.append({
                    "person_name": _clean_text(person.get("name")),
                    "id_type": _clean_text(id_type),
                    "url": url
                })

        people.append({
            "name": _clean_text(person.get("name")),
            "creator_order": person.get("creator_order"),
            "hydroshare_user_id": person.get("hydroshare_user_id"),
            "organization": _clean_text(person.get("organization")),
            "email": _clean_text(person.get("email")),
            "homepage": _clean_text(person.get("homepage")),
            "address": _clean_text(person.get("address")),
            "phone": _clean_text(person.get("phone")),
            "identifiers": person_identifiers
        })

    return people, identifiers_flat


def _parse_xml_semantics(xml_path: str) -> Dict[str, Any]:
    semantics = {
        "resource_xml_type": None,
        "typed_relations": [],
        "geospatial_relations": [],
        "collection_summary": None,
        "tool_config": None
    }

    if not os.path.exists(xml_path):
        return semantics

    tree = ET.parse(xml_path)
    root = tree.getroot()

    resource_elem = None
    for child in root:
        qn = _qname(child.tag)
        if qn in {"hsterms:CompositeResource", "hsterms:CollectionResource", "hsterms:ToolResource"}:
            resource_elem = child
            semantics["resource_xml_type"] = qn.replace("hsterms:", "")
            break

    if resource_elem is None:
        return semantics

    typed_relations = []
    for relation_container in resource_elem.findall(f"{{{NS['dc']}}}relation"):
        desc = next(iter(relation_container), None)
        if desc is None:
            continue

        for rel_elem in desc:
            target, rdf_object_kind = _elem_value(rel_elem)
            if not target:
                continue

            embedded_urls = _extract_urls(target)

            typed_relations.append({
                "predicate_qname": _qname(rel_elem.tag),
                "predicate_uri": _predicate_uri(rel_elem.tag),
                "target": target,
                "rdf_object_kind": rdf_object_kind,
                "target_looks_like_url": _looks_like_url(target),
                "embedded_urls": embedded_urls,
                "contains_embedded_url": len(embedded_urls) > 0,
                "target_resource_id": _extract_hydroshare_resource_id(target),
                "source_file": "raw_metadata.xml"
            })

    seen_rel = set()
    dedup_relations = []
    for rel in typed_relations:
        key = (rel["predicate_qname"], rel["target"])
        if key not in seen_rel:
            seen_rel.add(key)
            dedup_relations.append(rel)
    semantics["typed_relations"] = dedup_relations

    geospatial_relations = []
    for geo_container in resource_elem.findall(f"{{{NS['hsterms']}}}geospatialRelation"):
        desc = next(iter(geo_container), None)
        if desc is None:
            continue

        relation_name = None
        target = None
        rdf_object_kind = None
        target_pred_qname = None
        target_pred_uri = None

        for child in desc:
            child_qname = _qname(child.tag)

            if child_qname == "hsterms:relation_name":
                relation_name = _clean_text("".join(child.itertext()))
            else:
                value, kind = _elem_value(child)
                if value:
                    target = value
                    rdf_object_kind = kind
                    target_pred_qname = child_qname
                    target_pred_uri = _predicate_uri(child.tag)

        if target:
            embedded_urls = _extract_urls(target)

            geospatial_relations.append({
                "name": relation_name,
                "relation_category": "geospatial",
                "predicate_qname": target_pred_qname,
                "predicate_uri": target_pred_uri,
                "target": target,
                "rdf_object_kind": rdf_object_kind,
                "target_looks_like_url": _looks_like_url(target),
                "embedded_urls": embedded_urls,
                "contains_embedded_url": len(embedded_urls) > 0,
                "target_resource_id": _extract_hydroshare_resource_id(target),
                "source_file": "raw_metadata.xml"
            })

    seen_geo = set()
    dedup_geo = []
    for geo in geospatial_relations:
        key = (geo["name"], geo["target"])
        if key not in seen_geo:
            seen_geo.add(key)
            dedup_geo.append(geo)
    semantics["geospatial_relations"] = dedup_geo

    if semantics["resource_xml_type"] == "CollectionResource":
        members = []
        described_by = []

        for rel in semantics["typed_relations"]:
            if rel["predicate_qname"] == "dcterms:hasPart":
                members.append({
                    "citation_or_reference": rel["target"],
                    "member_resource_id": rel["target_resource_id"]
                })
            elif rel["predicate_qname"] == "hsterms:isDescribedBy":
                described_by.append(rel["target"])

        semantics["collection_summary"] = {
            "member_count": len(members),
            "members": members,
            "described_by": _clean_list(described_by)
        }

    if semantics["resource_xml_type"] == "ToolResource":
        app_home_page_url, _ = _find_hsterms_value(resource_elem, "AppHomePageUrl")
        request_url_base, _ = _find_hsterms_value(resource_elem, "RequestUrlBase")
        request_url_base_file, _ = _find_hsterms_value(resource_elem, "RequestUrlBaseFile")
        tool_version, _ = _find_hsterms_value(resource_elem, "ToolVersion")
        testing_protocol_url, _ = _find_hsterms_value(resource_elem, "TestingProtocolUrl")

        tool_icon_parent = resource_elem.find(f"{{{NS['hsterms']}}}ToolIcon")
        tool_icon_url = None
        tool_icon_data_url = None
        if tool_icon_parent is not None:
            icon_value_elem = tool_icon_parent.find(f".//{{{NS['hsterms']}}}value")
            icon_data_elem = tool_icon_parent.find(f".//{{{NS['hsterms']}}}data_url")
            tool_icon_url, _ = _elem_value(icon_value_elem)
            tool_icon_data_url, _ = _elem_value(icon_data_elem)

        tool_icon_url = _clean_text(tool_icon_url)
        tool_icon_data_url = _clean_text(tool_icon_data_url)
        tool_icon_data_url_summary = _summarize_data_url(tool_icon_data_url)

        tool_config = {
            "app_home_page_url": _clean_text(app_home_page_url),
            "request_url_base": _normalize_tool_url_template(request_url_base),
            "request_url_base_file": _normalize_tool_url_template(request_url_base_file),
            "tool_version": _clean_text(tool_version),
            "testing_protocol_url": _clean_text(testing_protocol_url),
            "tool_icon_url": tool_icon_url,
            "tool_icon_data_url_summary": tool_icon_data_url_summary,
            "supported_file_extensions": _find_hsterms_values(resource_elem, "SupportedFileExtensions"),
            "supported_resource_types": _find_hsterms_values(resource_elem, "SupportedResTypes"),
            "supported_sharing_status": _find_hsterms_values(resource_elem, "SupportedSharingStatus")
        }

        if any([
            tool_config["app_home_page_url"],
            tool_config["request_url_base"],
            tool_config["request_url_base_file"],
            tool_config["tool_version"],
            tool_config["testing_protocol_url"],
            tool_config["tool_icon_url"],
            tool_config["tool_icon_data_url_summary"]["present"],
            tool_config["supported_file_extensions"],
            tool_config["supported_resource_types"],
            tool_config["supported_sharing_status"]
        ]):
            semantics["tool_config"] = tool_config

    return semantics


def _build_relations_from_meta(meta_relations_raw: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    relations = []
    for rel in meta_relations_raw:
        if not isinstance(rel, dict):
            continue

        target = _clean_text(rel.get("value"))
        label = _clean_text(rel.get("type"))

        if not target:
            continue

        embedded_urls = _extract_urls(target)

        relations.append({
            "relation_label": label,
            "target": target,
            "rdf_object_kind": None,
            "target_looks_like_url": _looks_like_url(target),
            "embedded_urls": embedded_urls,
            "contains_embedded_url": len(embedded_urls) > 0,
            "target_resource_id": _extract_hydroshare_resource_id(target),
            "source_file": "meta_dump.json"
        })
    return relations


def _merge_relations(meta_relations: List[Dict[str, Any]], typed_relations: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    meta_labels_by_target = defaultdict(list)
    for rel in meta_relations:
        if rel["target"] and rel.get("relation_label"):
            meta_labels_by_target[rel["target"]].append(rel["relation_label"])

    merged = []
    seen = set()

    for rel in typed_relations:
        key = (rel.get("predicate_qname"), rel.get("target"))
        if key in seen:
            continue
        seen.add(key)

        merged.append({
            "predicate_qname": rel.get("predicate_qname"),
            "predicate_uri": rel.get("predicate_uri"),
            "relation_labels_from_meta": _clean_list(meta_labels_by_target.get(rel.get("target"), [])),
            "target": rel.get("target"),
            "rdf_object_kind": rel.get("rdf_object_kind"),
            "target_looks_like_url": rel.get("target_looks_like_url"),
            "embedded_urls": rel.get("embedded_urls", []),
            "contains_embedded_url": rel.get("contains_embedded_url", False),
            "target_resource_id": rel.get("target_resource_id"),
            "source_files": ["raw_metadata.xml"] + (["meta_dump.json"] if rel.get("target") in meta_labels_by_target else [])
        })

    xml_any_target = {rel.get("target") for rel in typed_relations}

    for rel in meta_relations:
        target = rel.get("target")
        label = rel.get("relation_label")

        if not target:
            continue

        if target not in xml_any_target:
            key = (None, target, label)
            if key in seen:
                continue
            seen.add(key)

            merged.append({
                "predicate_qname": None,
                "predicate_uri": None,
                "relation_labels_from_meta": [label] if label else [],
                "target": target,
                "rdf_object_kind": rel.get("rdf_object_kind"),
                "target_looks_like_url": rel.get("target_looks_like_url"),
                "embedded_urls": rel.get("embedded_urls", []),
                "contains_embedded_url": rel.get("contains_embedded_url", False),
                "target_resource_id": rel.get("target_resource_id"),
                "source_files": ["meta_dump.json"]
            })

    return merged


def _build_hydroshare_links(related_resources: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    links = []
    seen = set()

    for rel in related_resources:
        target_resource_id = rel.get("target_resource_id")
        if not target_resource_id:
            continue

        key = (rel.get("predicate_qname"), target_resource_id, rel.get("target"))
        if key in seen:
            continue
        seen.add(key)

        links.append({
            "predicate_qname": rel.get("predicate_qname"),
            "predicate_uri": rel.get("predicate_uri"),
            "target_resource_id": target_resource_id,
            "target": rel.get("target")
        })

    return links


def build_kg_node_from_disk(base_path: str, resource_id: str) -> Dict[str, Any]:
    """
    Transforms local Level-0 HydroShare artifacts into a structured JSON node
    suitable as input for KG extraction. Works offline.
    """
    res_dir = os.path.join(base_path, resource_id)

    if not os.path.isdir(res_dir):
        return {"resource_id": resource_id, "error": f"Directory not found: {res_dir}"}

    try:
        meta = _read_json(os.path.join(res_dir, "meta_dump.json"), {})
        sys_meta = _read_json(os.path.join(res_dir, "sys_meta_dump.json"), {})
        files_inventory = _read_json(os.path.join(res_dir, "file_inventory.json"), [])

        xml_path = os.path.join(res_dir, "raw_metadata.xml")
        xml_semantics = _parse_xml_semantics(xml_path)

        documentation = _read_optional_readme(res_dir)

        creators, creator_identifiers = _build_people(meta.get("creators", []))
        contributors, contributor_identifiers = _build_people(meta.get("contributors", []))

        relations_from_meta = _build_relations_from_meta(meta.get("relations", []))
        related_resources = _merge_relations(relations_from_meta, xml_semantics["typed_relations"])
        hydroshare_links = _build_hydroshare_links(related_resources)

        rights_data = meta.get("rights", {}) or {}
        spatial = meta.get("spatial_coverage")
        period = meta.get("period_coverage")

        ext_counter = Counter(
            (_clean_text(f.get("extension", "")).lower() if isinstance(f, dict) else "")
            for f in files_inventory
            if isinstance(f, dict) and f.get("extension")
        )

        has_geospatial_files = any(ext in GEOSPATIAL_EXTENSIONS for ext in ext_counter.keys())

        dataset_node = {
            "resource_id": resource_id,
            "resource_type": meta.get("type") or sys_meta.get("resource_type") or xml_semantics["resource_xml_type"] or "CompositeResource",
            "title": _clean_text(meta.get("title") or sys_meta.get("resource_title")),
            "abstract": _clean_text(meta.get("abstract") or sys_meta.get("abstract")),
            "url": _clean_text(meta.get("url") or sys_meta.get("resource_url") or f"https://www.hydroshare.org/resource/{resource_id}/"),
            "identifier": _clean_text(meta.get("identifier")),
            "citation": _clean_text(meta.get("citation")),
            "language": _clean_text(meta.get("language")),

            "creators": creators,
            "contributors": contributors,
            "creator_identifiers": creator_identifiers,
            "contributor_identifiers": contributor_identifiers,

            "rights": {
                "statement": _clean_text(rights_data.get("statement")),
                "url": _clean_text(rights_data.get("url"))
            } if rights_data else None,

            "awards": [
                {
                    "title": _clean_text(a.get("title")),
                    "number": _clean_text(a.get("number")),
                    "funding_agency_name": _clean_text(a.get("funding_agency_name")),
                    "funding_agency_url": _clean_text(a.get("funding_agency_url"))
                }
                for a in meta.get("awards", [])
                if isinstance(a, dict)
            ],

            "additional_metadata": meta.get("additional_metadata", {}),

            "system_metadata": {
                "doi": _clean_text(sys_meta.get("doi")),
                "date_created": _clean_text(sys_meta.get("date_created")),
                "date_last_updated": _clean_text(sys_meta.get("date_last_updated")),
                "bag_last_downloaded": _clean_text(sys_meta.get("bag_last_downloaded")),
                "published": sys_meta.get("published", False),
                "immutable": sys_meta.get("immutable", False),
                "public": sys_meta.get("public", False),
                "discoverable": sys_meta.get("discoverable", False),
                "shareable": sys_meta.get("shareable", False),
                "bag_url": _clean_text(sys_meta.get("bag_url")),
                "science_metadata_url": _clean_text(sys_meta.get("science_metadata_url")),
                "resource_map_url": _clean_text(sys_meta.get("resource_map_url")),
                "resource_url": _clean_text(sys_meta.get("resource_url")),
                "content_types": sys_meta.get("content_types", [])
            },

            "subjects": _clean_list(meta.get("subjects", [])),

            "spatial_coverage": {
                "type": _clean_text(spatial.get("type")),
                "northlimit": spatial.get("northlimit") or spatial.get("north"),
                "southlimit": spatial.get("southlimit") or spatial.get("south"),
                "eastlimit": spatial.get("eastlimit") or spatial.get("east"),
                "westlimit": spatial.get("westlimit") or spatial.get("west"),
                "units": _normalize_units(spatial.get("units")) if isinstance(spatial, dict) else None,
                "projection": _clean_text(spatial.get("projection")) if isinstance(spatial, dict) else None
            } if isinstance(spatial, dict) else None,

            "temporal_coverage": {
                "start": _clean_text(period.get("start")),
                "end": _clean_text(period.get("end"))
            } if isinstance(period, dict) else None,

            "relations_from_meta": relations_from_meta,
            "typed_relations": xml_semantics["typed_relations"],
            "related_resources": related_resources,
            "hydroshare_links": hydroshare_links,

            "geospatial_relations": xml_semantics["geospatial_relations"],
            "collection_summary": xml_semantics["collection_summary"],
            "tool_config": xml_semantics["tool_config"],

            "documentation": documentation,

            "files": files_inventory,
            "file_summary": {
                "total_files": len(files_inventory),
                "extensions": dict(sorted(ext_counter.items())),
                "has_readme_file": any(
                    isinstance(f, dict) and _clean_text(f.get("file_name", "")).lower().startswith("readme")
                    for f in files_inventory
                )
            },

            "data_services": {
                "wms": f"https://geoserver.hydroshare.org/geoserver/HS-{resource_id}/wms?request=GetCapabilities",
                "wcs": f"https://geoserver.hydroshare.org/geoserver/HS-{resource_id}/wcs?request=GetCapabilities"
            } if has_geospatial_files else None,

            "relation_summary": {
                "meta_relation_count": len(relations_from_meta),
                "typed_relation_count": len(xml_semantics["typed_relations"]),
                "merged_relation_count": len(related_resources),
                "hydroshare_link_count": len(hydroshare_links),
                "geospatial_relation_count": len(xml_semantics["geospatial_relations"])
            }
        }

        return dataset_node

    except Exception as e:
        print(f"Offline Extraction Failure [Resource: {resource_id}]: {str(e)}")
        return {"resource_id": resource_id, "error": str(e)}

In [ ]:
test_id = "7477452a93e04b1099a3879353c2253a" 

extracted_json = build_kg_node_from_disk(base_path="./ciroh_corpus", resource_id=test_id)

#print(json.dumps(extracted_json, indent=4))
print(json.dumps(extracted_json["documentation"], indent=4, ensure_ascii=False))

In [17]:
# Define file paths
csv_file_path = 'HydroShare_IDs_03272026.csv'
base_corpus_path = './ciroh_corpus'
master_kg_output_file = 'ciroh_hydroshare_corpus.json'

# 1. Read and validate IDs from the CSV file
ciroh_corpus_ids = []
invalid_ids = []
seen_ids = set()

if not os.path.exists(csv_file_path):
    raise FileNotFoundError(f"File not found: {csv_file_path}")

with open(csv_file_path, 'r', encoding='utf-8-sig') as file:
    for line in file:
        clean_id = line.strip().lower()
        if not clean_id:
            continue
        if len(clean_id) == 32:
            if clean_id not in seen_ids:
                ciroh_corpus_ids.append(clean_id)
                seen_ids.add(clean_id)
        else:
            invalid_ids.append(clean_id)

print(f"Starting Offline Transformation for {len(ciroh_corpus_ids)} resources...\n")
print("-" * 50)

if invalid_ids:
    print(f"Warning: {len(invalid_ids)} invalid IDs were skipped.")
    for bad_id in invalid_ids:
        print(f" - {bad_id}")
    print("-" * 50)

# 2. Orchestrate the offline transformation
knowledge_graph_nodes = []
failed_transformations = []

for index, res_id in enumerate(ciroh_corpus_ids, start=1):
    print(f"[{index}/{len(ciroh_corpus_ids)}] Extracting semantics for: {res_id}")

    node_data = build_kg_node_from_disk(base_path=base_corpus_path, resource_id=res_id)

    if "error" in node_data:
        print(f"  -> Transformation Error: {node_data['error']}")
        failed_transformations.append(res_id)
    else:
        knowledge_graph_nodes.append(node_data)

# 3. Persist the Master JSON Corpus
if knowledge_graph_nodes:
    try:
        with open(master_kg_output_file, 'w', encoding='utf-8') as outfile:
            json.dump(knowledge_graph_nodes, outfile, indent=4, ensure_ascii=False)
        print(f"\nMaster KG Corpus saved successfully: {master_kg_output_file}")
    except Exception as e:
        print(f"\nError saving Master Corpus: {str(e)}")
else:
    print("\nWarning: No KG nodes were generated. Output file was not created.")

# 4. Execution reporting
print("\n" + "=" * 50)
print("KG TRANSFORMATION REPORT")
print("=" * 50)
print(f"Total valid IDs processed: {len(ciroh_corpus_ids)}")
print(f"Successfully transformed nodes: {len(knowledge_graph_nodes)}")
print(f"Failed transformations: {len(failed_transformations)}")

if failed_transformations:
    print("\nList of IDs that failed transformation:")
    for fid in failed_transformations:
        print(f" - {fid}")

Starting Offline Transformation for 42 resources...

--------------------------------------------------
[1/42] Extracting semantics for: 7d960b7fdfee480895fd845bade1b75a
[2/42] Extracting semantics for: 0ef4366e7711478fa2637f5049b4881a
[3/42] Extracting semantics for: 3200bab682ec4c3287147cbe40e768ee
[4/42] Extracting semantics for: a8b243957f7946e388d10ab206990675
[5/42] Extracting semantics for: 88e0ebf2719c492381efcb27fba71032
[6/42] Extracting semantics for: 18134d8103d84deeb2a9a5008068158d
[7/42] Extracting semantics for: 1a2e115c212f4f4a80660f94339205e6
[8/42] Extracting semantics for: 7477452a93e04b1099a3879353c2253a
[9/42] Extracting semantics for: 0d0a8d67a7fe4676bc84f6973dde0fc8
[10/42] Extracting semantics for: fc8539358fe64ca6a47468728a0687a1
[11/42] Extracting semantics for: 48b3355015f84e98a2cc6b6d134e77f2
[12/42] Extracting semantics for: 023bfa101586432ba0f6ed9ddfea60a9
[13/42] Extracting semantics for: 23b51cb99b5445a1af740a21c5acaecb
[14/42] Extracting semantics for: 

In [69]:
'''
extracted_data = extract_kg_metadata(hs_session, test_id)
        
# Serialize to verify JSON compatibility
print(json.dumps(extracted_data, indent=4))
'''

'\nextracted_data = extract_kg_metadata(hs_session, test_id)\n\n# Serialize to verify JSON compatibility\nprint(json.dumps(extracted_data, indent=4))\n'